# 02. Error Analysis — 학습 후 OOF 분석
> **여기만 바꾸면 된다** → `CONFIG` 블록의 경로
> 학습이 끝날 때마다 재실행

In [ ]:
# =============================================
# ★ 여기만 수정 ★
# =============================================
CONFIG = dict(
    exp_name     = 'exp001_baseline',
    oof_prob_dir = 'outputs/exp001_baseline',   # oof_fold*.npy 있는 폴더
    train_csv    = 'input/train_metadata.csv',
    audio_dir    = 'input/train_audio',
    label_col    = 'primary_label',
    id_col       = 'filename',
    sample_rate  = 32000,
    threshold    = 0.5,
    top_k_error  = 15,    # 오분류 많은 클래스 top K 표시
    n_listen     = 3,     # 클래스당 청취할 오분류 샘플 수
)
# =============================================

import warnings; warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import librosa
import librosa.display
import IPython.display as ipd
from pathlib import Path
from sklearn.metrics import (classification_report, confusion_matrix,
                              f1_score, roc_auc_score, average_precision_score)
from tqdm.auto import tqdm

sns.set_theme(style='whitegrid')
print('라이브러리 로드 완료')

## 1. OOF 예측값 로드

In [ ]:
out_dir = Path(CONFIG['oof_prob_dir'])
oof_files = sorted(out_dir.glob('oof_fold*.npy'))
label_files = sorted(out_dir.glob('oof_labels*.npy'))

print(f'발견된 OOF 파일: {[f.name for f in oof_files]}')

# fold 별로 쌓기
all_probs  = np.concatenate([np.load(f) for f in oof_files],  axis=0)
all_labels = np.concatenate([np.load(f) for f in label_files], axis=0)

print(f'OOF prob shape : {all_probs.shape}')
print(f'OOF label shape: {all_labels.shape}')

# 라벨맵 로드
import json
label_map_path = out_dir.parent / 'input' / 'label_map.json'
if not label_map_path.exists():
    label_map_path = Path('input/label_map.json')
with open(label_map_path) as f:
    label_map = json.load(f)
idx2label = {v: k for k, v in label_map.items()}
class_names = [idx2label[i] for i in range(len(idx2label))]
print(f'클래스 수: {len(class_names)}')

## 2. 전체 지표 요약

In [ ]:
thr = CONFIG['threshold']
all_preds = (all_probs > thr).astype(int)

# multilabel인지 single-label인지 자동 판단
is_multilabel = all_labels.sum(axis=1).max() > 1

if is_multilabel:
    macro_f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    micro_f1 = f1_score(all_labels, all_preds, average='micro', zero_division=0)
    try:
        auc = roc_auc_score(all_labels, all_probs, average='macro')
        cmap = average_precision_score(all_labels, all_probs, average='macro')
    except:
        auc = cmap = float('nan')
    print(f'Macro F1 : {macro_f1:.4f}')
    print(f'Micro F1 : {micro_f1:.4f}')
    print(f'Macro AUC: {auc:.4f}')
    print(f'cmAP     : {cmap:.4f}')
else:
    y_true_1d = all_labels.argmax(axis=1)
    y_pred_1d = all_preds.argmax(axis=1)
    acc = (y_true_1d == y_pred_1d).mean()
    macro_f1 = f1_score(y_true_1d, y_pred_1d, average='macro', zero_division=0)
    print(f'Accuracy : {acc:.4f}')
    print(f'Macro F1 : {macro_f1:.4f}')

## 3. Per-class F1 (약한 클래스 찾기)

In [ ]:
per_class_f1 = f1_score(all_labels, all_preds, average=None, zero_division=0)
per_class_ap = average_precision_score(all_labels, all_probs, average=None)

cls_df = pd.DataFrame({
    'class'     : class_names,
    'f1'        : per_class_f1,
    'ap'        : per_class_ap,
    'n_pos'     : all_labels.sum(axis=0).astype(int),
    'n_pred'    : all_preds.sum(axis=0).astype(int),
}).sort_values('f1')

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# 하위 K
worst = cls_df.head(CONFIG['top_k_error'])
axes[0].barh(worst['class'], worst['f1'], color='tomato', alpha=0.8)
axes[0].set_title(f'F1 하위 {CONFIG["top_k_error"]} 클래스')
axes[0].set_xlabel('F1 Score')

# F1 vs 샘플 수
axes[1].scatter(cls_df['n_pos'], cls_df['f1'], alpha=0.5, color='steelblue')
axes[1].set_xlabel('학습 샘플 수')
axes[1].set_ylabel('F1 Score')
axes[1].set_title('샘플 수 vs F1 (상관관계 확인)')

plt.tight_layout(); plt.show()

print('\n--- F1=0 클래스 (완전히 못 맞힘) ---')
zero_f1 = cls_df[cls_df['f1'] == 0]
if len(zero_f1):
    print(zero_f1[['class','n_pos','n_pred']].to_string(index=False))
    print('→ 샘플 수 부족 or 클래스가 너무 어려움 → oversampling or class_weight 고려')
else:
    print('없음 ✓')

## 4. Confusion Matrix (자주 헷갈리는 클래스 쌍)

In [ ]:
# 클래스 수가 많으면 상위 오분류 쌍만 추출
from itertools import combinations

# 각 샘플에서 가장 높은 확률 클래스를 예측으로 사용 (single-label 방식으로 분석)
y_pred_top = all_probs.argmax(axis=1)
y_true_top = all_labels.argmax(axis=1)

# 오분류 행렬 상위 20쌍
wrong_mask = y_pred_top != y_true_top
wrong_true = y_true_top[wrong_mask]
wrong_pred = y_pred_top[wrong_mask]

confusion_pairs = pd.Series(
    [f'{class_names[t]} → {class_names[p]}' for t, p in zip(wrong_true, wrong_pred)]
).value_counts().head(20)

fig, ax = plt.subplots(figsize=(10, 6))
confusion_pairs.sort_values().plot.barh(ax=ax, color='coral', alpha=0.8)
ax.set_title('자주 발생하는 오분류 쌍 Top 20')
ax.set_xlabel('오분류 횟수')
plt.tight_layout(); plt.show()

print('\n→ 자주 헷갈리는 쌍: 두 클래스의 오디오를 직접 비교해서 feature 차이 확인')

## 5. 신뢰도 분포 (높은 확률로 틀리는 케이스)

In [ ]:
max_probs = all_probs.max(axis=1)
is_correct = (y_pred_top == y_true_top)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(max_probs[is_correct],  bins=40, alpha=0.6, color='steelblue', label='정답')
axes[0].hist(max_probs[~is_correct], bins=40, alpha=0.6, color='tomato',    label='오답')
axes[0].set_title('최대 확률 분포 (정답 vs 오답)')
axes[0].set_xlabel('max probability')
axes[0].legend()

# 높은 확률로 틀린 케이스 (hard error)
high_conf_wrong = (~is_correct) & (max_probs > 0.8)
axes[1].text(0.1, 0.5,
    f'높은 확률(>0.8)로 틀린 샘플\n'
    f'{high_conf_wrong.sum()}개 ({high_conf_wrong.mean()*100:.1f}%)\n\n'
    f'→ 라벨 오류 또는 매우 유사한 클래스',
    transform=axes[1].transAxes, fontsize=12, va='center')
axes[1].axis('off')

plt.tight_layout(); plt.show()

# 높은 확률로 틀린 샘플 목록
df = pd.read_csv(CONFIG['train_csv'])
print(f'높은 확률로 틀린 샘플 Top 10:')
hard_idx = np.where(high_conf_wrong)[0]
for idx in hard_idx[:10]:
    true_cls = class_names[y_true_top[idx]]
    pred_cls = class_names[y_pred_top[idx]]
    prob = max_probs[idx]
    print(f'  [{idx}] 정답={true_cls}, 예측={pred_cls}, 확률={prob:.3f}')

## 6. Threshold 탐색 (OOF 기반)

In [ ]:
thresholds = np.arange(0.1, 0.91, 0.05)
f1_scores = []
for thr in thresholds:
    preds = (all_probs > thr).astype(int)
    f1 = f1_score(all_labels, preds, average='macro', zero_division=0)
    f1_scores.append(f1)

best_thr = thresholds[np.argmax(f1_scores)]
best_f1  = max(f1_scores)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(thresholds, f1_scores, marker='o', color='steelblue')
ax.axvline(best_thr, color='red', linestyle='--', label=f'최적 threshold: {best_thr:.2f}')
ax.set_title('Global Threshold vs Macro F1')
ax.set_xlabel('Threshold'); ax.set_ylabel('Macro F1')
ax.legend()
plt.tight_layout(); plt.show()

print(f'현재 threshold ({CONFIG["threshold"]}): F1 = {f1_scores[np.argmin(np.abs(thresholds - CONFIG["threshold"]))]: .4f}')
print(f'최적 threshold ({best_thr:.2f}):        F1 = {best_f1:.4f}')
print(f'\n→ configs/exp001_baseline.yaml의 inference.threshold를 {best_thr:.2f}로 업데이트 권장')

## 7. 오분류 샘플 직접 청취
> 가장 자주 틀리는 클래스의 오분류 샘플을 들어본다

In [ ]:
# 가장 F1이 낮은 클래스 3개의 오분류 샘플 청취
df_meta = pd.read_csv(CONFIG['train_csv'])
worst_classes = cls_df.head(3)['class'].tolist()

for target_cls in worst_classes:
    cls_idx = label_map[target_cls]
    # 이 클래스가 정답인데 틀린 샘플
    wrong_idx = np.where((y_true_top == cls_idx) & (y_pred_top != cls_idx))[0]
    
    print(f'\n=== {target_cls} (F1={per_class_f1[cls_idx]:.3f}) — 오분류 {len(wrong_idx)}개 ===')
    
    for i, idx in enumerate(wrong_idx[:CONFIG['n_listen']]):
        try:
            file_id = df_meta.iloc[idx][CONFIG['id_col']]
            path = Path(CONFIG['audio_dir']) / file_id
            wav, sr = librosa.load(path, sr=CONFIG['sample_rate'], duration=5.0)
            pred_cls = class_names[y_pred_top[idx]]
            print(f'  [{i+1}] 예측={pred_cls} (확률={all_probs[idx, y_pred_top[idx]]:.3f})')
            print(f'       파일: {file_id}')
            display(ipd.Audio(wav, rate=sr))

            fig, axes = plt.subplots(1, 2, figsize=(12, 2))
            axes[0].plot(wav, lw=0.5, color='steelblue')
            axes[0].set_title('파형'); axes[0].set_yticks([])
            mel = librosa.feature.melspectrogram(y=wav, sr=sr, n_mels=64)
            mel_db = librosa.power_to_db(mel, ref=np.max)
            librosa.display.specshow(mel_db, sr=sr, x_axis='time', y_axis='mel', ax=axes[1])
            axes[1].set_title('Mel-spectrogram')
            plt.tight_layout(); plt.show()
        except Exception as e:
            print(f'  로드 실패: {e}')

## 8. 분석 요약 → 다음 실험 방향

In [ ]:
print('='*55)
print(f'Error Analysis 요약 — {CONFIG["exp_name"]}')
print('='*55)
print(f'Macro F1        : {macro_f1:.4f}')
print(f'최적 Threshold  : {best_thr:.2f} (현재: {CONFIG["threshold"]})')
print(f'F1=0 클래스 수  : {len(zero_f1)}개')
print(f'고신뢰 오분류   : {high_conf_wrong.sum()}개')
print()
print('--- 다음 실험 후보 ---')
if len(zero_f1) > 5:
    print('① F1=0 클래스 많음 → oversampling / class_weight / focal loss')
if high_conf_wrong.sum() > 50:
    print('② 고신뢰 오분류 많음 → 라벨 노이즈 확인 / label_smoothing')
if abs(best_thr - CONFIG['threshold']) > 0.1:
    print(f'③ threshold를 {best_thr:.2f}로 업데이트')
print('④ 자주 헷갈리는 클래스 쌍 확인 → Section 4 참고')
print('='*55)